<a href="https://colab.research.google.com/github/hursoo/big_k-modern_1/blob/main/gb_071_topic_extract_basic(big1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.기본설정 및 패키지
- 토픽모델링의 기본과정 개관
- 기본 파라미터 설정 등은 혼란을 줄이기 위해 gb_061에서도 사용했고, 본 연구의 최종 토픽으로 선택한 num_topics=8, seed=7, alpha=0.05, eta=0.1로 설정. (물론 이 설정은 다르게 해도 기본과정 이해에는 무방하다.)

In [1]:
# =================================================================
# 셀 1: 라이브러리 설치
# =================================================================
# 설치 과정 표시, 에러 출력은 숨김

!pip install -q tomotopy==0.13.0 numpy==1.23.5 kneed 2> /dev/null # 2> /dev/null : 에러 출력 메시지를 숨깁니다.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 13.4 MB/s eta 0:00:00


=> "런타임 / 세션 다시 시작"

In [1]:
# =================================================================
# 셀 2: 기본 설정 및 라이브러리 임포트
# =================================================================
# "세션 다시 시작" 후에 이 셀부터 실행합니다.

# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')
print("\n✅ 구글 드라이브 마운트 완료!")

# 경로 지정
file_path = '/content/drive/MyDrive/big_km_history01/'

Mounted at /content/drive

✅ 구글 드라이브 마운트 완료!


In [2]:
import sys
import os, re
import pandas as pd
import numpy as np
import random
import warnings

from tomotopy import DMRModel
from tomotopy import TermWeight
from tomotopy import utils
import tomotopy as tp

import matplotlib.pyplot as plt

# 2.입력 데이터 불러오기

In [3]:
# gb_031에서 생성한 '기본 데이터프레임' 불러오기

gb_df = pd.read_excel(file_path + 'result/gb_data_2(doc,1g2g,wn_cls).xlsx')
gb_df

,doc_id,doc_raw,doc_split_12gram,r_no,title,w_new,ho_no,grid_1,wn_cls
0,1,創刊辭 强者도 부르짖고 弱者도 부르짖으며 優者도 부르짖고 劣者도 부르짖도다 東西南北...,창간 辭 강자 약자 優者 劣者 동서 남북 사해 팔방 소리 소리 판단 좌우 間 다수 ...,1,創刊辭,uk01,1,01q,0
1,2,哲人은 말하되 多數 人民의 聲은 곳 神의 聲이라 하엿나니 神은 스스로 要求가 없는지...,哲人 다수 인민 요구 인민 소리 요구 발표 갈앙 인민 소리 갈앙 다수 인민 갈앙 요...,1,創刊辭,uk01,1,01q,0
2,3,世界를 알라 사람은 天使도 안이며 野獸도 안이오 오즉 사람일 뿐이로다 이만치 進化된...,세계 사람 야수 사람 진화 진화 지식 진화 도덕 동물 세계 천당 지옥 세계 진화 국...,2,世界를 알라,uk01,1,01q,0
3,4,사람과 世界는 決코 논하볼 것이 안이엇다 사람으로 된 世界 世界로 된 사람 둘이 안...,사람 세계 사람 세계 세계 사람 세계 대표 시대 가치 사람 대표 문화 상징 符號 사...,2,世界를 알라,uk01,1,01q,0
4,5,過去는 論할 것이 업도다만은 今日과 가티 交通이 이마마하고 知識이 이마마하고 一切의...,過去 금일 교통 지식 일체 문물 오늘 세계 理解 공자 천하 천하 理解 오늘 날 사람...,2,世界를 알라,uk01,1,01q,0
...,...,...,...,...,...,...,...,...,...
6797,6798,미국에서는 每人에게 奴僕이 식 돌아감니다 단 機械奴僕만 이건 무슨 말이냐 하면 기계...,미국 노복 機械 노복 機械 力 운전 계산 人力 계산 미국 국민 사람 機械 輸入 숫자...,334,유로빠와 아메리카(一) 금년 봄에 모쓰크바 엑쓰페리멘탈 劇場에서 한 「트로츠끼」의 講演.,김철산,72,24q,2
6798,6799,매년 저축은ㅡ필요의 비용을 다 쓴 뒤의 것 말이지요ㅡ매년에 억 딸라 金 루불로는 억...,저축 필요 비용 미국 교과 書 실상 미국 富豪 캐나다 영국 미국 부분 미국 캐나다 ...,334,유로빠와 아메리카(一) 금년 봄에 모쓰크바 엑쓰페리멘탈 劇場에서 한 「트로츠끼」의 講演.,김철산,72,24q,2
6799,6800,그러고 또 캐나다는 아조 겸손스럽게 좀 유순하게 미국의 北邊繼續이라고ㅡ국제연맹의 축...,캐나다 미국 국제_연맹 국제_연맹 원인 경제 機械 캐나다 산업 북미 자본 점령 부분...,334,유로빠와 아메리카(一) 금년 봄에 모쓰크바 엑쓰페리멘탈 劇場에서 한 「트로츠끼」의 講演.,김철산,72,24q,2
6800,6801,25년 전에는 영국이 美보다 배나 더 되게 輸入하엿섯음니다 캐나다 사람들은 지금 그...,영국 미국 輸入 캐나다 사람 영국 一部 미국 호주 캐나다 진화 호주 일본 침입 보호...,334,유로빠와 아메리카(一) 금년 봄에 모쓰크바 엑쓰페리멘탈 劇場에서 한 「트로츠끼」의 講演.,김철산,72,24q,2


# 3.실행 코드(사용자 함수들)

In [5]:
# 1) OpenMP 스레드 수 제한 (결과 재현성을 위해)
os.environ["OMP_NUM_THREADS"] = "1"

# 2) 파이썬 자체 random, numpy 랜덤 시드 고정
def set_global_seeds(seed_value=1000):
    random.seed(seed_value)
    np.random.seed(seed_value)

# 3) `a_data`를 `metadata`로 변환하는 함수
def transform_a_data_to_metadata(misc: dict):
    return {'metadata': str(misc['a_data'])}

def run_dmr_model(gridL, lineL, num_topics=10, seed=100, iterations=1000, alpha=0.1, eta=0.01):
    """
    DMR 모델 실행 및 메타데이터 저장.
    """
    # 파이썬 랜덤, numpy 랜덤 시드 고정
    set_global_seeds(seed)

    print(f"\nTraining DMR Model with {num_topics} topics, alpha={alpha}, eta={eta}...")

    # DMR 모델 초기화 (tomotopy 내부 시드 설정)
    model = DMRModel(
        k=num_topics,
        seed=seed,
        tw=TermWeight.ONE,
        alpha=alpha,
        eta=eta
    )
    corpus = utils.Corpus()

    # 코퍼스에 문서 추가 (lineL은 이미 단어 리스트로 전처리되었다고 가정)
    for grid, tokens in zip(gridL, lineL):
        corpus.add_doc(tokens, a_data=grid)

    # 모델에 코퍼스 추가 (메타데이터 변환 포함)
    model.add_corpus(corpus, transform=transform_a_data_to_metadata)

    # 학습
    for i in range(0, iterations, 20):  # 20단위로 학습 반복
        model.train(20, workers=1)
        print(f"Iteration: {i + 20}\tLog-likelihood: {model.ll_per_word:.4f}")

    # --- ✨ 변경점: 토픽별 단어 수를 직접 계산 ---
    # DMRModel에는 count_by_topics 속성이 없으므로,
    # 모든 문서의 단어-토픽 할당을 기반으로 직접 계산합니다.
    topic_counts = [0] * model.k
    for doc in model.docs:
        for topic_idx in doc.topics:
            topic_counts[topic_idx] += 1

    print("\n<Topics>")
    # 각 토픽의 정보(단어 수, 상위 단어)를 가져와서 형식에 맞게 출력
    for i in range(model.k):
        # 직접 계산한 토픽별 단어 수 가져오기
        topic_word_count = topic_counts[i]

        # 토픽별 상위 단어 20개 가져오기
        top_words = model.get_topic_words(i, top_n=20)

        # 단어만 추출하여 공백으로 연결
        word_list = [word[0] for word in top_words]
        words_str = ' '.join(word_list)

        # 최종 형식으로 출력
        print(f"| #{i} ({topic_word_count}) : {words_str}")

    # 원래 코드와의 호환성을 위해 topics 리스트도 반환
    topics = [model.get_topic_words(i, top_n=20) for i in range(model.k)]

    return model, topics

# 4.코드 실행

In [6]:
# --- 사용 예시 ---

# 1. 메타데이터 준비
gridL = gb_df['grid_1'].tolist()

# 2. 텍스트 데이터 준비 (각 문서를 단어 리스트로 변환)
lineL = gb_df['doc_split_12gram'].apply(lambda x: str(x).split()).tolist()

# 3. 모델 실행
model, topics = run_dmr_model(
    gridL,
    lineL,
    num_topics=8,
    seed=7, ####
    iterations=1000,
    alpha=0.05,
    eta=0.1
)


Training DMR Model with 8 topics, alpha=0.05, eta=0.1...
Iteration: 20	Log-likelihood: -7.8370
Iteration: 40	Log-likelihood: -7.7045
Iteration: 60	Log-likelihood: -7.6746
Iteration: 80	Log-likelihood: -7.6570
Iteration: 100	Log-likelihood: -7.6406
Iteration: 120	Log-likelihood: -7.6298
Iteration: 140	Log-likelihood: -7.6241
Iteration: 160	Log-likelihood: -7.6265
Iteration: 180	Log-likelihood: -7.6206
Iteration: 200	Log-likelihood: -7.6148
Iteration: 220	Log-likelihood: -7.6125
Iteration: 240	Log-likelihood: -7.6115
Iteration: 260	Log-likelihood: -7.6079
Iteration: 280	Log-likelihood: -7.6026
Iteration: 300	Log-likelihood: -7.6080
Iteration: 320	Log-likelihood: -7.6051
Iteration: 340	Log-likelihood: -7.6025
Iteration: 360	Log-likelihood: -7.6056
Iteration: 380	Log-likelihood: -7.6066
Iteration: 400	Log-likelihood: -7.6027
Iteration: 420	Log-likelihood: -7.6039
Iteration: 440	Log-likelihood: -7.6023
Iteration: 460	Log-likelihood: -7.6023
Iteration: 480	Log-likelihood: -7.6020
Iteration:

# The End of Note